# Лабораторная работа №2: Классификация цветов автомобилей (DVM)

**Цель:** решить задачу классификации цвета автомобиля по фронтальным изображениям из DVM и сравнить:

1. Классификатор, обученный **с нуля** (реализация ResNet-подобной сети вручную).
2. Аналогичный классификатор, **предобученный на ImageNet**, с последующим fine-tuning.

Метрика качества: `F1_macro` (целевое требование: `F1_macro > 0.8`).


## 0) Подготовка данных

1. Скачайте с сайта DVM архив **Quality checked front-view images (730 MB)**.
2. Распакуйте архив в папку `data/raw/dvm_front`.
3. Ожидаемая структура директорий:

```
data/raw/dvm_front/
  Brand/
    Model/
      Year/
        Color/
          *.jpg
```

Цвет берется из названия папки `Color`.


In [ ]:
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tv_models

sns.set_theme(style='whitegrid')

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
@dataclass
class Config:
    data_root: str = 'data/raw/dvm_front'
    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 2
    epochs_scratch: int = 18
    epochs_pretrained: int = 12
    lr_scratch: float = 1e-3
    lr_pretrained: float = 3e-4
    weight_decay: float = 1e-4
    min_samples_per_class: int = 200
    dropout_p: float = 0.2
    fast_dev_run: bool = False

cfg = Config()
cfg


In [ ]:
def build_index(data_root: str) -> pd.DataFrame:
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(
            f'Папка {data_root} не найдена. Сначала скачайте и распакуйте DVM front-view.'
        )

    rows = []
    for img_path in data_root.rglob('*.jpg'):
        parts = img_path.parts
        if len(parts) < 5:
            continue
        color = parts[-2].strip().lower()
        rows.append({'path': str(img_path), 'color': color})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError('Не найдено ни одного .jpg файла в data_root')

    return df


def normalize_color_name(c: str) -> str:
    c = c.lower().strip()
    aliases = {
        'grey': 'gray',
        'silver metallic': 'silver',
        'blue metallic': 'blue',
        'red metallic': 'red',
        'white pearl': 'white',
        'black metallic': 'black',
    }
    return aliases.get(c, c)


def prepare_dataframe(cfg: Config) -> pd.DataFrame:
    df = build_index(cfg.data_root)
    df['color'] = df['color'].map(normalize_color_name)

    counts = df['color'].value_counts()
    keep_classes = counts[counts >= cfg.min_samples_per_class].index
    df = df[df['color'].isin(keep_classes)].copy()

    if cfg.fast_dev_run:
        df = (
            df.groupby('color', group_keys=False)
              .apply(lambda x: x.sample(min(len(x), 250), random_state=42))
              .reset_index(drop=True)
        )

    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    return df


df = prepare_dataframe(cfg)
print('Размер датасета после фильтрации:', len(df))
print('Число классов:', df['color'].nunique())
print(df['color'].value_counts().head(15))


In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(x=df['color'].value_counts().index, y=df['color'].value_counts().values)
plt.xticks(rotation=45, ha='right')
plt.title('Распределение классов после фильтрации')
plt.ylabel('Количество изображений')
plt.tight_layout()
plt.show()


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['color'],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['color'],
)

label2id = {c: i for i, c in enumerate(sorted(train_df['color'].unique()))}
id2label = {i: c for c, i in label2id.items()}

for d in (train_df, val_df, test_df):
    d['label'] = d['color'].map(label2id)

print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))
print('Классы:', label2id)


In [ ]:
class DVMColorDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        label = int(row['label'])
        return image, label


train_tfms = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = DVMColorDataset(train_df, transform=train_tfms)
val_ds = DVMColorDataset(val_df, transform=val_tfms)
test_ds = DVMColorDataset(test_df, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)


## 1) Модель с нуля: ResNet-подобная архитектура (ручная реализация)


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1, downsample=None, dropout_p=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.dropout = nn.Dropout2d(dropout_p) if dropout_p > 0 else nn.Identity()

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class SmallResNet(nn.Module):
    def __init__(self, num_classes, dropout_p=0.0):
        super().__init__()
        self.in_ch = 64

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.layer1 = self._make_layer(64, blocks=2, stride=1, dropout_p=dropout_p)
        self.layer2 = self._make_layer(128, blocks=2, stride=2, dropout_p=dropout_p)
        self.layer3 = self._make_layer(256, blocks=2, stride=2, dropout_p=dropout_p)
        self.layer4 = self._make_layer(512, blocks=2, stride=2, dropout_p=dropout_p)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_ch, blocks, stride, dropout_p):
        downsample = None
        if stride != 1 or self.in_ch != out_ch:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

        layers = [BasicBlock(self.in_ch, out_ch, stride=stride, downsample=downsample, dropout_p=dropout_p)]
        self.in_ch = out_ch
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_ch, out_ch, stride=1, downsample=None, dropout_p=dropout_p))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


## Общие функции обучения и оценки


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []
    y_true_all, y_pred_all = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        preds = logits.argmax(dim=1)
        y_true_all.extend(yb.cpu().numpy().tolist())
        y_pred_all.extend(preds.cpu().numpy().tolist())

    f1 = f1_score(y_true_all, y_pred_all, average='macro')
    return float(np.mean(losses)), float(f1)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    losses = []
    y_true_all, y_pred_all = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        losses.append(loss.item())
        preds = logits.argmax(dim=1)
        y_true_all.extend(yb.cpu().numpy().tolist())
        y_pred_all.extend(preds.cpu().numpy().tolist())

    f1 = f1_score(y_true_all, y_pred_all, average='macro')
    return float(np.mean(losses)), float(f1), y_true_all, y_pred_all


def fit_model(model, train_loader, val_loader, lr, weight_decay, epochs, patience=4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}
    best_state = None
    best_val_f1 = -1.0
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_f1 = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_f1, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_f1)

        history['train_loss'].append(tr_loss)
        history['train_f1'].append(tr_f1)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)

        print(f'Epoch {epoch:02d}/{epochs} | train_loss={tr_loss:.4f} train_f1={tr_f1:.4f} | val_loss={val_loss:.4f} val_f1={val_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print('Early stopping triggered')
            break

    model.load_state_dict(best_state)
    return model, history, best_val_f1


def plot_history(hist, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(hist['train_loss'], label='train')
    axes[0].plot(hist['val_loss'], label='val')
    axes[0].set_title(f'{title}: Loss')
    axes[0].legend()

    axes[1].plot(hist['train_f1'], label='train')
    axes[1].plot(hist['val_f1'], label='val')
    axes[1].set_title(f'{title}: F1_macro')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
num_classes = len(label2id)

scratch_model = SmallResNet(num_classes=num_classes, dropout_p=cfg.dropout_p)
scratch_model, scratch_hist, scratch_best_val_f1 = fit_model(
    model=scratch_model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=cfg.lr_scratch,
    weight_decay=cfg.weight_decay,
    epochs=cfg.epochs_scratch,
    patience=4,
)

plot_history(scratch_hist, 'Scratch SmallResNet')
print('Best val F1 (scratch):', round(scratch_best_val_f1, 4))


## 2) Предобученная модель: ResNet18 (ImageNet) + fine-tuning


In [ ]:
weights = tv_models.ResNet18_Weights.IMAGENET1K_V1
pretrained_model = tv_models.resnet18(weights=weights)
pretrained_model.fc = nn.Linear(pretrained_model.fc.in_features, num_classes)

pretrained_model, pretrained_hist, pretrained_best_val_f1 = fit_model(
    model=pretrained_model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=cfg.lr_pretrained,
    weight_decay=cfg.weight_decay,
    epochs=cfg.epochs_pretrained,
    patience=4,
)

plot_history(pretrained_hist, 'Pretrained ResNet18')
print('Best val F1 (pretrained):', round(pretrained_best_val_f1, 4))


In [ ]:
criterion = nn.CrossEntropyLoss()

scratch_test_loss, scratch_test_f1, y_true_s, y_pred_s = evaluate(scratch_model, test_loader, criterion)
pre_test_loss, pre_test_f1, y_true_p, y_pred_p = evaluate(pretrained_model, test_loader, criterion)

results_df = pd.DataFrame([
    {
        'model': 'Scratch SmallResNet',
        'test_loss': scratch_test_loss,
        'test_f1_macro': scratch_test_f1,
    },
    {
        'model': 'Pretrained ResNet18 (fine-tuned)',
        'test_loss': pre_test_loss,
        'test_f1_macro': pre_test_f1,
    },
])

results_df


In [ ]:
best_name = results_df.sort_values('test_f1_macro', ascending=False).iloc[0]['model']
print('Лучшая модель по test F1_macro:', best_name)

if best_name.startswith('Pretrained'):
    y_true_best, y_pred_best = y_true_p, y_pred_p
else:
    y_true_best, y_pred_best = y_true_s, y_pred_s

print('\nClassification report для лучшей модели:')
print(classification_report(y_true_best, y_pred_best, target_names=[id2label[i] for i in range(num_classes)]))


In [ ]:
cm = confusion_matrix(y_true_best, y_pred_best)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_norm,
    cmap='Blues',
    xticklabels=[id2label[i] for i in range(num_classes)],
    yticklabels=[id2label[i] for i in range(num_classes)],
)
plt.title('Нормализованная confusion matrix (лучшая модель)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Автоматически сформированные выводы

In [ ]:
scratch_f1 = float(results_df.loc[results_df['model'] == 'Scratch SmallResNet', 'test_f1_macro'].iloc[0])
pre_f1 = float(results_df.loc[results_df['model'] == 'Pretrained ResNet18 (fine-tuned)', 'test_f1_macro'].iloc[0])

print(f'1) F1_macro модели с нуля: {scratch_f1:.4f}')
print(f'2) F1_macro предобученной модели: {pre_f1:.4f}')

if pre_f1 > scratch_f1:
    print('3) Лучше сработал transfer learning (предобученная модель).')
elif pre_f1 < scratch_f1:
    print('3) Лучше сработала модель, обученная с нуля.')
else:
    print('3) Модели показали одинаковое качество.')

best_f1 = max(scratch_f1, pre_f1)
print(f'4) Порог F1_macro > 0.8: {"достигнут" if best_f1 > 0.8 else "не достигнут"}.')
print('5) Анализ конкретных ошибок по цветам смотрите в confusion matrix и classification report выше.')
